# QuantumSynapse Fabric - Qiskit Test Cases (Colab)

Every cell below is copied from `notebooks/_verify_before_packaging.py` in the project repo, which was **actually executed** before this notebook was assembled - not written directly into `.ipynb` and assumed correct. Run top to bottom; each cell asserts its own expected result, so a green run-through *is* the test passing, not just "no red text."

**What this demonstrates, and what it doesn't**, stated once here rather than per-cell:
- QRNG and GHZ entanglement are genuine Qiskit circuit simulation (Born-rule sampling on a classical simulator) - not hardware entropy, not a claim of quantum advantage.
- BB84 is a correct simulation of the protocol's math, including *why* it detects eavesdropping (the QBER jump below is a real consequence of measurement disturbance in the simulated statevector) - it provides **no real security** over this notebook's connection to Colab's servers, because there's no physical quantum channel involved. See `docs/QuantumSynapse-Fabric-DPR-Complete.md` section 3 for the longer version of why that's a hard physics limit, not an engineering gap.

This notebook is standalone (no FastAPI server, no repo checkout needed) - just Qiskit.

In [ ]:
!pip install -q qiskit==2.5.2 qiskit-aer==0.17.2

In [ ]:
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
print("AerSimulator ready.")

## Test case 1: Quantum RNG (Hadamard superposition)

In [ ]:
def qrng(num_bits: int) -> str:
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits))
    qc.measure(range(num_bits), range(num_bits))
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1).result()
    return list(result.get_counts().keys())[0]

bits = qrng(8)
print("QRNG result:", bits, "-> integer", int(bits, 2))
assert len(bits) == 8 and set(bits) <= {"0", "1"}
print("PASS")

## Test case 2: GHZ entanglement - every shot should be fully correlated

In [ ]:
def ghz_counts(num_qubits: int, shots: int) -> dict:
    qc = QuantumCircuit(num_qubits, num_qubits)
    qc.h(0)
    for i in range(1, num_qubits):
        qc.cx(0, i)
    qc.measure(range(num_qubits), range(num_qubits))
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=shots).result()
    return result.get_counts()

counts = ghz_counts(4, 500)
print("GHZ counts:", counts)
assert all(len(set(k)) == 1 for k in counts), "found an uncorrelated outcome - would mean a real bug"
print("PASS: all", sum(counts.values()), "shots were fully correlated (all-0s or all-1s only)")

## Test case 3: BB84 - QBER should jump when an eavesdropper is simulated
Same logic as `quantum-core/main.py`'s `/v1/quantum/bb84/simulate` endpoint, inlined here so this cell needs nothing but Qiskit.

In [ ]:
def run_single_shot(qc):
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])

def prepare_and_measure_bit(alice_bit, alice_basis, bob_basis, eve_intercepts):
    qc = QuantumCircuit(1, 1)
    if alice_bit == 1:
        qc.x(0)
    if alice_basis == "X":
        qc.h(0)
    if eve_intercepts:
        eve_basis = random.choice(["Z", "X"])
        if eve_basis == "X":
            qc.h(0)
        qc.measure(0, 0)
        eve_bit = run_single_shot(qc)
        qc = QuantumCircuit(1, 1)
        if eve_bit == 1:
            qc.x(0)
        if eve_basis == "X":
            qc.h(0)
    if bob_basis == "X":
        qc.h(0)
    qc.measure(0, 0)
    return run_single_shot(qc)

def bb84(num_bits, with_eve):
    alice_bits = [random.randint(0, 1) for _ in range(num_bits)]
    alice_bases = [random.choice(["Z", "X"]) for _ in range(num_bits)]
    bob_bases = [random.choice(["Z", "X"]) for _ in range(num_bits)]
    bob_bits = [prepare_and_measure_bit(alice_bits[i], alice_bases[i], bob_bases[i], with_eve) for i in range(num_bits)]
    sifted = [i for i in range(num_bits) if alice_bases[i] == bob_bases[i]]
    mismatches = sum(1 for i in sifted if alice_bits[i] != bob_bits[i])
    return (mismatches / len(sifted)) * 100 if sifted else 0

qber_clean = bb84(150, with_eve=False)
qber_eve = bb84(150, with_eve=True)
print(f"QBER without eavesdropper: {qber_clean:.1f}%")
print(f"QBER with eavesdropper:    {qber_eve:.1f}%")
assert qber_clean < 5, f"clean QBER too high: {qber_clean}"
assert qber_eve > 11, f"eavesdropper QBER should be well above the 11% abort threshold: {qber_eve}"
print("PASS: eavesdropper correctly pushes QBER past the detection threshold")

## Summary
If every cell above printed `PASS`, this Colab environment reproduces the same three test cases already verified in `quantum-core/main.py` against a live FastAPI instance (see `docs/DEVELOPMENT.md` sections 5 and 11). This notebook doesn't call that service - it's an independent, standalone re-derivation using only Qiskit, which is itself a useful cross-check: if this notebook and the live service ever disagreed, that would flag a real bug worth investigating, not something to average away.